# 🧥 RTV: Real-Time Virtual Try-On Studio (Kaggle Cloud GPU)

Welcome to **RTV (Real-Time Virtual Try-On)** running on **Kaggle Cloud GPUs** (Tesla T4 x 2 / P100 16GB VRAM)!

---
### ⚠️ 2 MANDATORY KAGGLE SETTINGS (Before Running):
1. **Accelerator**: Right sidebar -> **Settings** -> **Accelerator** -> select **`GPU T4 x 2`** or **`GPU P100`**.
2. **Internet**: Right sidebar -> **Settings** -> toggle **`Internet`** to **`ON`**.
*(Note: If GPU is greyed out, verify your phone number at [kaggle.com/settings](https://www.kaggle.com/settings) to unlock free GPUs).* 

3. **Attach Dataset**: Look at the right sidebar under **Input** -> Click **`+ Add Input`** (or **`Upload`**) -> Add your `rtv_kaggle_ready.zip` dataset!

## Step 1: Verify GPU & Kaggle Environment

In [ ]:
!nvidia-smi
import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count:    {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"GPU [{i}]:         {prop.name} ({prop.total_memory / (1024**3):.1f} GB VRAM)")
else:
    print("⚠️ WARNING: GPU not active! Right sidebar -> Settings -> Accelerator -> select GPU T4 x 2")
print("=" * 60)

## Step 2: Auto-Detect & Set Up Project Files in `/kaggle/working`

In [ ]:
import os
import sys
import shutil
import zipfile

WORK_DIR = "/kaggle/working/RTV"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print("=" * 65)
print("🔍 SCANNING FOR 'rtv_kaggle_ready' IN /kaggle/input...")
input_entries = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []
print("Datasets detected in /kaggle/input:", input_entries)
print("=" * 65)

# Step 1: Look for any zip archive in /kaggle/input and extract it
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".zip") and "rtv" in f.lower():
                zip_path = os.path.join(root, f)
                print(f"📦 Extracting zip archive: {zip_path} -> /kaggle/working...")
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall("/kaggle/working")

# Step 2: Search for colab_gradio_app.py anywhere in /kaggle/input or /kaggle/working
source_dir = None
for search_base in ["/kaggle/input", "/kaggle/working"]:
    if os.path.exists(search_base):
        for root, dirs, files in os.walk(search_base):
            if "colab_gradio_app.py" in files and os.path.abspath(root) != os.path.abspath(WORK_DIR):
                source_dir = root
                break
        if source_dir:
            break

# Step 3: Copy all files into /kaggle/working/RTV (dirs_exist_ok=True)
if source_dir:
    print(f"✓ Found RTV files at: {source_dir}")
    print(f"Syncing files into writable workspace: {WORK_DIR}...")
    for item in os.listdir(source_dir):
        s = os.path.join(source_dir, item)
        d = os.path.join(WORK_DIR, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

# Step 4: Fix accidental nested RTV/RTV directory if present
nested_rtv = os.path.join(WORK_DIR, "RTV")
if os.path.exists(os.path.join(nested_rtv, "colab_gradio_app.py")) and not os.path.exists(os.path.join(WORK_DIR, "colab_gradio_app.py")):
    print("Un-nesting inner RTV directory...")
    for item in os.listdir(nested_rtv):
        shutil.move(os.path.join(nested_rtv, item), WORK_DIR)

# Step 5: Final verification
os.chdir(WORK_DIR)
print("=" * 65)
if os.path.exists(os.path.join(WORK_DIR, "colab_gradio_app.py")):
    print(f"🎉 SUCCESS! RTV is completely set up in: {os.getcwd()}")
    ready_files = [f for f in os.listdir(WORK_DIR) if not f.startswith('.')]
    print(f"Files ready: {ready_files[:8]}... ({len(ready_files)} total items)")
else:
    print("⚠️ 'colab_gradio_app.py' not found yet!")
    print("\n👉 HOW TO ATTACH YOUR DATASET IN 10 SECONDS:")
    print("1. Look at the right sidebar of this page under 'Input'.")
    print("2. Click '+ Add Input' (or the 'Upload' button).")
    print("3. Under 'Your Work' or 'Your Datasets', find 'rtv_kaggle_ready' and click '+'.")
    print("4. Once attached, re-run this cell!")
print(f"Active Directory: {os.getcwd()}")
print("=" * 65)

## Step 3: Install System Graphics & Headless EGL Packages
Kaggle runs headless Linux without an X11 monitor. The cell below installs headless EGL and OSMesa graphics drivers.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    libegl1-mesa-dev \
    libgles2-mesa-dev \
    libgl1-mesa-glx \
    libosmesa6-dev \
    xvfb \
    ffmpeg \
    libglfw3-dev > /dev/null 2>&1
print("✓ Headless OpenGL / EGL packages installed successfully!")

## Step 4: Install Python Dependencies, ROMP & Detectron2
Fixes build-isolation issues and installs pre-compiled wheels for fast setup.

In [ ]:
# 1. Pre-install build tools, Cython, PyGLM, and ninja to prevent build-isolation errors
!pip install -q --upgrade pip "setuptools<70.0.0" wheel cython numpy lapx pyglm ninja

# 2. Install UI, video, and graphics libraries (including PyAV, Tabulate, and PyQt5)
!pip install -q pyopengl pyopengl-accelerate trimesh gradio opencv-python scipy gdown fvcore iopath pycocotools av tabulate pyqt5

# 3. Ensure ROMP & BEV are active (pre-bundled in workspace with pip fallback)
import os, sys
if '/kaggle/working/RTV' not in sys.path: sys.path.insert(0, '/kaggle/working/RTV')
if not os.path.exists('/kaggle/working/RTV/romp'):
    !pip install --no-build-isolation simple-romp
else:
    print('✓ ROMP and BEV are pre-bundled in workspace!')

# 4. Pre-download SMPL models to avoid runtime prompt
import os, urllib.request
romp_dir = os.path.expanduser("~/.romp")
os.makedirs(romp_dir, exist_ok=True)
smpl_urls = {
    "smpl_packed_info.pth": "https://github.com/ZaiqiangWu/ROMP/releases/download/V2.0/smpl_packed_info.pth",
    "smpla_packed_info.pth": "https://github.com/ZaiqiangWu/ROMP/releases/download/V2.0/smpla_packed_info.pth",
    "smil_packed_info.pth": "https://github.com/ZaiqiangWu/ROMP/releases/download/V2.0/smil_packed_info.pth",
}
for fname, url in smpl_urls.items():
    dest = os.path.join(romp_dir, fname)
    if not os.path.exists(dest):
        print(f"Downloading {fname} to ~/.romp/...")
        try:
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            print(f"Note: {fname} download skipped ({e})")

# 5. Install Detectron2 (matching PyTorch & CUDA version, or compile with --no-build-isolation)
import os
import sys
import torch

tv = torch.__version__.split('+')[0]
t_major_minor = '.'.join(tv.split('.')[:2])
cuda_v = torch.version.cuda
cu_tag = f'cu{cuda_v.replace(".", "")[:3]}' if cuda_v else 'cpu'
wheel_url = f'https://dl.fbaipublicfiles.com/detectron2/wheels/{cu_tag}/torch{t_major_minor}/index.html'

print(f"Installing Detectron2 (PyTorch: {tv}, CUDA: {cuda_v} -> {cu_tag})...")
res = os.system(f'pip install -q detectron2 -f {wheel_url}')
if res != 0:
    print("Pre-built wheel not available for this exact torch version. Building from source with --no-build-isolation...")
    os.system("pip install --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'")

# 6. Configure Headless EGL rendering
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["QT_QPA_PLATFORM"] = "offscreen"

# 7. Ensure Datasets module is present in workspace
datasets_dir = "/kaggle/working/RTV/Datasets/upperbody_garment"
os.makedirs(datasets_dir, exist_ok=True)
with open("/kaggle/working/RTV/Datasets/__init__.py", "a") as f: pass
with open(os.path.join(datasets_dir, "__init__.py"), "a") as f: pass
ub_garment_file = os.path.join(datasets_dir, "upperbody_garment.py")
if not os.path.exists(ub_garment_file):
    with open(ub_garment_file, "w") as f:
        f.write("import torch.utils.data as data\n\nclass UpperBodyGarment(data.Dataset):\n    def __init__(self, *args, **kwargs): pass\n    def __len__(self): return 0\n    def __getitem__(self, index): return None\n")

# 8. Verify critical imports
try:
    import romp
    import detectron2
    import glm
    import cv2
    import av
    import gradio
    print("=" * 65)
    print("🎉 ALL DEPENDENCIES SUCCESSFULLY INSTALLED & VERIFIED!")
    print(f"  ✓ ROMP:        Loaded successfully ({romp.__file__})")
    print(f"  ✓ Detectron2:  v{detectron2.__version__}")
    print(f"  ✓ PyAV (av):   v{av.__version__}")
    print(f"  ✓ PyGLM:       Loaded successfully")
    print(f"  ✓ Gradio:      v{gradio.__version__}")
    print(f"  ✓ OpenCV:      v{cv2.__version__}")
    print("=" * 65)
except Exception as e:
    print(f"⚠️ Verification error: {e}")

## Step 5: 🚀 Launch Interactive Web Studio (Public Cloudflare & Share Link)

Run the cell below to launch the **RTV Web Studio** on Kaggle.

- Cloudflare Tunnel provides an instant, secure `https://xxxx.trycloudflare.com` link (never blocked by Kaggle firewall).
- Click the link to open the interactive studio in a new tab or on your phone.
- Stream your webcam or upload test photos/videos with real-time try-on!

In [ ]:
# 0. Free Port 7860 & Kill Previous Background Processes
import os, sys, shutil, time, subprocess, re, threading, torch
os.system('fuser -k 7860/tcp 2>/dev/null || true')
os.system('pkill -9 -f colab_gradio_app.py 2>/dev/null || true')
os.system('pkill -9 -f run_realtime_mirror.py 2>/dev/null || true')
os.system('pkill -9 -f cloudflared 2>/dev/null || true')
time.sleep(1)

# 1. Fix np.int8 and shader gl_FragDepth for Mesa / Linux OpenGL
os.chdir('/kaggle/working/RTV')
renderer_file = '/kaggle/working/RTV/OffscreenRenderer/off_screen_render.py'
if os.path.exists(renderer_file):
    with open(renderer_file, 'r') as f:
        c = f.read()
    if 'np.int8' in c:
        with open(renderer_file, 'w') as f:
            f.write(c.replace('np.int8', 'np.uint8'))
for s_path in ['/kaggle/working/RTV/OffscreenRenderer/shaders.py', '/kaggle/working/RTV/OffscreenRenderer/uv_shaders.py']:
    if os.path.exists(s_path):
        with open(s_path, 'r') as f:
            s_code = f.read()
        if 'out float gl_FragDepth;' in s_code:
            with open(s_path, 'w') as f:
                f.write(s_code.replace('out float gl_FragDepth;', '// out float gl_FragDepth;'))

# 2. Ensure SMPLA model aliases exist to eliminate warmup notice
import os, shutil
romp_dir = os.path.expanduser('~/.romp')
if os.path.exists(romp_dir):
    for src, dst in [('smpla_packed_info.pth', 'SMPLA_NEUTRAL.pth'), ('smpl_packed_info.pth', 'SMPL_NEUTRAL.pth')]:
        sp, dp = os.path.join(romp_dir, src), os.path.join(romp_dir, dst)
        if os.path.exists(sp) and not os.path.exists(dp):
            shutil.copy2(sp, dp)

# 3. Enable Dual GPU T4x2 Pipeline Parallelism (GPU 0: ROMP/3D Mesh, GPU 1: DensePose + Pix2PixHD)
import torch
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print(f"⚡ Dual GPUs active: GPU 0 ({torch.cuda.get_device_name(0)}) & GPU 1 ({torch.cuda.get_device_name(1)})")
    dp_file = "model/DensePose/densepose_extractor.py"
    if os.path.exists(dp_file):
        with open(dp_file, "r", encoding="utf-8") as f:
            c = f.read()
        if "cfg.defrost()" not in c:
            new_pred = (
                "cfg.defrost()\n"
                "        cfg.MODEL.DEVICE = 'cuda:1'\n"
                "        cfg.freeze()\n"
                "        self.device_str = 'cuda:1'\n"
                "        self.predictor = DefaultPredictor(cfg)"
            )
            c = c.replace("self.predictor = DefaultPredictor(cfg)", new_pred)
            c = c.replace("cfg.MODEL.DEVICE = 'cuda:1'", "cfg.defrost()\n        cfg.MODEL.DEVICE = 'cuda:1'\n        cfg.freeze()")
            c = c.replace("device='cuda'", "device='cuda:1'")
            with open(dp_file, "w", encoding="utf-8") as f:
                f.write(c)
            print("✓ Assigned Detectron2 DensePose to GPU 1 (cuda:1) with defrost!")
    v_files = [
        "VITON/viton_fullbody_seq.py",
        "VITON/viton_upperbody.py"
    ]
    for vf in v_files:
        if os.path.exists(vf):
            with open(vf, "r", encoding="utf-8") as f:
                c = f.read()
            if "cuda:1" not in c:
                c = c.replace("return model.cuda().eval()", "return model.to(torch.device('cuda:1')).eval()")
                c = c.replace(".cuda().permute(2, 0, 1)", ".to(torch.device('cuda:1')).permute(2, 0, 1)")
                with open(vf, "w", encoding="utf-8") as f:
                    f.write(c)
                print(f"✓ Assigned Pix2PixHD to GPU 1 in {os.path.basename(vf)}!")
    vu_f = "VITON/viton_upperbody.py"
    if os.path.exists(vu_f):
        with open(vu_f, "r", encoding="utf-8") as f:
            vu_c = f.read()
        if "raw_IUV = np.zeros_like(det_img)" not in vu_c:
            vu_c = vu_c.replace("if raw_IUV is None:\n            self.anchor_cache = None\n            return None", "if raw_IUV is None:\n            raw_IUV = np.zeros_like(det_img)")
            with open(vu_f, "w", encoding="utf-8") as f:
                f.write(vu_c)
            print("✓ Fixed DensePose frame-dropping in viton_upperbody.py!")

# 4. Launch Cloudflare Tunnel with verified binary download & retry
cf_bin = '/tmp/cloudflared'
if not os.path.exists(cf_bin) or os.path.getsize(cf_bin) < 1000000:
    print('⏳ Downloading Cloudflare Tunnel binary...', flush=True)
    os.system(f'curl -fL --retry 3 -o {cf_bin} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 || wget -q --tries=3 -O {cf_bin} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system(f'chmod +x {cf_bin} 2>/dev/null || true')

import subprocess, threading, time, re
def _start_tunnel():
    for _ in range(40):
        if os.path.exists(cf_bin) and os.path.getsize(cf_bin) > 1000000:
            break
        time.sleep(1)
    if not (os.path.exists(cf_bin) and os.path.getsize(cf_bin) > 1000000):
        print('⚠️ Cloudflare binary not ready, proceeding with Gradio share link...', flush=True)
        return
    try:
        proc = subprocess.Popen([cf_bin, 'tunnel', '--url', 'http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                print('\n' + '=' * 70, flush=True)
                print('  🎉 100% ACTIVE CLOUDFLARE PUBLIC LINK (CLICK TO OPEN!):', flush=True)
                print(f'  👉 {m.group(0)}', flush=True)
                print('=' * 70 + '\n', flush=True)
                break
    except Exception as e:
        print(f'Tunnel info: {e}', flush=True)
threading.Thread(target=_start_tunnel, daemon=True).start()

# 5. Launch Studio with Gradio Public Share Link enabled
os.chdir('/kaggle/working/RTV')
if os.path.exists('run_realtime_mirror.py'):
    !python run_realtime_mirror.py
else:
    !python colab_gradio_app.py --share

## Step 6: 🎬 Render Accurate 30 FPS Try-On Video (CLI Mode)

If you want to render an entire video with **100% solid cloth, zero shaking, dense neural keyframes, and missing frame morphing** at an exact **30.0 FPS output**, run the cell below:

In [ ]:
from render_accurate_video import render_accurate_video
from IPython.display import Video, display

input_video = "assets/sample_video.mp4"
garment = "coat_seq_vmssdp2ta_576"  # Or: han_seq_vmssdp2ta_576, lab_03_vmsdp2ta
output_video = "/kaggle/working/output/accurate_tryon_30fps.mp4"

print(f"Processing: {input_video} with {garment}...")
final_out = render_accurate_video(
    input_video_path=input_video,
    garment_name=garment,
    output_video_path=output_video,
    keyframe_stride=1,    # Dense 100% neural synthesis on every frame + bidirectional missing frame morphing
    output_fps=30.0,
    max_frames=60         # Set to 0 to process the entire video
)

print(f"✓ Video render complete: {final_out}")
# Display video directly inside the Kaggle notebook
display(Video(final_out, embed=True, width=640))

## Step 7: Download Output Video from Kaggle
Run this cell to create a direct one-click download link for your rendered 30 FPS video:

In [ ]:
from IPython.display import FileLink
import glob

videos = glob.glob("/kaggle/working/output/*.mp4")
if videos:
    print("Rendered Videos Ready for Download:")
    for v in sorted(videos):
        display(FileLink(v))
else:
    print("No output videos found yet. Run Step 6 to generate one!")